ADTA5560 Recurrent Neural Network Assignment 3
Build, Train, and Test a Simple RNN on Sine Wave Data 
Follow the steps discussed in the lectures (PDFs and videos) and redo the lecture project of building, training, and testing a simple recurrent neural network on sine wave data using TensorFlow (backend) and Keras

##  Importing Libraries

In [ ]:
# import basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# for timeseries RNN neural network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

In [ ]:
# import keras: TimeseriesGenerator
# this class produces time series batches used on training/testing the model
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# import keras: MinMax scalar
# this class is used to preprocess (scale) the data
from sklearn.preprocessing import MinMaxScaler

# Generate Data 

In [ ]:
# create a simple sine wave in numpy

x = np.linspace(0,64,1024)
y = np.sin(x)


In [ ]:
x

In [ ]:
y

In [ ]:
plt.plot(x, y)

In [ ]:
# load data into a dataframe

df= pd.DataFrame(data=y, index=x, columns=['Sine'])
df.head(5)

In [ ]:
len(df)

# Split train and test data

In [ ]:
# set percentage of data used for testing

test_percent=0.2

In [ ]:
# Number of data points reserved for tesing the model
# 20% of the original dataset

len(df)*test_percent

In [ ]:
# need to find the length (number of data points) of the testing dataset
# It has been found (above) that around 205 data points are used for testing 

test_length = np.round(len(df)*test_percent)
test_length 

In [ ]:
# The testing dataset  starts at this index
# Index starting with 0

test_start_index = int(len(df) - test_length)

In [ ]:
test_start_index

In [ ]:
# Create separate training / testing datasets

# Training dataset: All the indices from start to the test_start_index 
# ( excluding the test_start_index)

data_train = df.iloc[: test_start_index]

# Testing dataset: all the indices from the test_start_index to the end of the dataframe
# (including test_start_index)

data_test= df.iloc[test_start_index :]

data_train.head(5)


In [ ]:
data_test.head(5)


## Normalize the data ( scale it into [0,1])

In [ ]:
# Create a MinMaxScalar to normalize the data
scalar = MinMaxScaler()

In [ ]:
# IGNORE the warning: Just converting the data to floats
# Only Scale the Trainig data  - Not scale the testing data
# train the sccalar to perform the normalization

scalar.fit(data_train)

In [ ]:
# Normalize the training dataset
normalized_train= scalar.transform(data_train)

# Normalize the testing dataset
normalized_train= scalar.transform(data_test)

# Create timeseries generator instance

In [ ]:
#from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# Set the length of the input sequence
# i.e., the number of time series steps used to predict the next one
# Use 50 historical data points to predict the next one
length = 50

# Set the batch size
# Number of time series samples in each batch
# Only one sample in each batch
batch_size = 1

# Create a TimeSeriesGenerator for training
# This time series generator produces time series batches used to train the model
# TimeseriesGenerator(inputs dataset, outputs dataset, length, batch_size)
train_tsGenerator50 = TimeseriesGenerator(normalized_train, normalized_train, length=length, batch_size=batch_size)

# Get the length of the normalized training data
len(normalized_train)


In [ ]:
# first batch
X,y=train_tsGenerator50[0]


In [ ]:
X.flatten()

In [ ]:
y

## Build Train and Test Model

# Build simple RNN Model

In [ ]:
# Data set: Only one column/attribute: Sine values of index x
# Features: How many features used to train the model: Only one
n_features = 1

# define model
model = Sequential()

# Add a simple RNN layer: Using SimpleRNN cells
# This layer has 100 neurons: One neuron for each input data point
# NOTES: # time series steps of the input sequence: 50
model.add(SimpleRNN(100, input_shape=(length, n_features)))

# Add a FC (fully-connected) layer for the final prediction
# Only one neuron of the Dense/Fully-Connected layer
# --> Output: Predict the next data point of the input sequence: only one value
model.add(Dense(1))


# complie model

In [ ]:
# complie the model

model.compile(optimizer='adam', loss='mse')
model.summary()


# Train (Fit) Model

In [ ]:
# Fit the model
# Use fit_generator(), Not fit()

model.fit_generator(train_tsGenerator50, epochs=5)


# Visualize Model's Porformance After Training  

In [ ]:
# load the loss data into a dataframe

df_model_loss = pd.DataFrame(model.history.history)

# visualize the loss data using Dataframe.plot()
df_model_loss.plot()


# Evaluate Model on Test Data

# A sneakpeak into the test data

In [ ]:
length

In [ ]:
first_eval_batch = normalized_train[-length : ]
first_eval_batch


In [ ]:
first_eval_batch=first_eval_batch.reshape((1,length,n_features))

first_eval_batch

In [ ]:
first_eval_batch.shape


# Evaluate Model

In [ ]:
# ALL the code for evaluation

# Declare a list to store all the predictions
# Similar to: test_predictions = list()
test_predictions = []

# Get the first time series batch for testing
# The 1st batch: The 1st time series input sequence
# = The last 50 data points of the train data set
first_eval_batch = normalized_train[-length:]

# Reshape the batch into 3D array: #samples/batch x Length x #features
current_batch = first_eval_batch.reshape((1, length, n_features))

# Run a FOR loop to make a prediction for each batch
for i in range(len(data_test)):

    # Get the value of the first element: The prediction
    current_pred = model.predict(current_batch)[0]

    # Store prediction into the list of predictions
    test_predictions.append(current_pred)

    # Generate a new batch to prepare for the next iteration of testing
    # --> Drop the first data point of the current input sequence
    current_batch = np.append(current_batch[:,1,:],[[current_pred]],axis=1)


In [ ]:
# convert the scaled result back to the real values
true_predictions= scalar.inverse_transform(test_predictions)

true_predictions

In [ ]:
data_test

In [ ]:
# Copy the true values of predictions into the dataframe of original test data
# Add it as another column

data_test['Predictions'] = true_predictions

In [ ]:
data_test

In [ ]:
# visualize the updated data set
# Compare the predicted sine wave against the original sine wave

data_test.plot(figsize=(12,8))
